# PDF Report Generation

Converted from `src/generate_pdf.py`

---

**Beschreibung:** PDF-Generator fur Projektdokumentation
Converts Markdown to PDF with embedded plots

In [ ]:
import markdown
from pathlib import Path
import base64


def embed_images_in_html(html_content, plots_dir):
    """Ersetzt Bild-Referenzen durch base64-eingebettete Bilder."""
    plots_path = Path(plots_dir)
    
    # Find all image references
    import re
    img_pattern = r'<img[^>]*src="([^"]*)"[^>]*>'
    
    def replace_img(match):
        src = match.group(1)
        img_path = plots_path / Path(src).name
        
        if img_path.exists():
            with open(img_path, 'rb') as f:
                img_data = base64.b64encode(f.read()).decode()
            return f'<img src="data:image/png;base64,{img_data}" style="max-width:100%; height:auto;">'
        return match.group(0)
    
    return re.sub(img_pattern, replace_img, html_content)


def md_to_pdf(md_path, pdf_path, plots_dir):
    """Konvertiert Markdown zu PDF."""
    print(f"  Lese: {md_path}")
    
    with open(md_path, 'r', encoding='utf-8') as f:
        md_content = f.read()
    
    # Ersetze Plot-Pfade
    md_content = md_content.replace('](plots/', f']({plots_dir}/')
    
    # Markdown zu HTML
    html_content = markdown.markdown(
        md_content,
        extensions=['tables', 'fenced_code', 'toc']
    )
    
    # CSS fur PDF (KEINE Kursivschrift!)
    css = """
    <style>
        @page {
            size: A4;
            margin: 2cm;
        }
        body {
            font-family: Arial, Helvetica, sans-serif;
            font-size: 11pt;
            line-height: 1.5;
            color: #333;
        }
        /* KEINE KURSIVSCHRIFT */
        em, i {
            font-style: normal !important;
            font-weight: normal;
        }
        h1 {
            color: #2c3e50;
            border-bottom: 2px solid #3498db;
            padding-bottom: 10px;
            page-break-before: always;
            font-size: 20pt;
        }
        h1:first-of-type {
            page-break-before: avoid;
        }
        h2 {
            color: #34495e;
            border-bottom: 1px solid #bdc3c7;
            padding-bottom: 5px;
            font-size: 16pt;
            margin-top: 20px;
        }
        h3 {
            color: #7f8c8d;
            font-size: 13pt;
            margin-top: 15px;
        }
        table {
            border-collapse: collapse;
            width: 100%;
            margin: 15px 0;
        }
        th, td {
            border: 1px solid #bdc3c7;
            padding: 8px 12px;
            text-align: left;
        }
        th {
            background-color: #3498db;
            color: white;
            font-weight: bold;
        }
        tr:nth-child(even) {
            background-color: #f8f9fa;
        }
        code {
            background-color: #f4f4f4;
            padding: 2px 6px;
            border-radius: 3px;
            font-family: 'Courier New', monospace;
            font-size: 10pt;
        }
        pre {
            background-color: #f4f4f4;
            padding: 15px;
            border-radius: 5px;
            overflow-x: auto;
            font-size: 9pt;
        }
        img {
            max-width: 100%;
            height: auto;
            display: block;
            margin: 20px auto;
            border: 1px solid #ddd;
            border-radius: 5px;
            box-shadow: 0 2px 5px rgba(0,0,0,0.1);
        }
        ul, ol {
            margin-left: 20px;
        }
        li {
            margin-bottom: 5px;
        }
        hr {
            border: none;
            border-top: 2px solid #3498db;
            margin: 30px 0;
        }
        .highlight {
            background-color: #fff3cd;
            padding: 10px;
            border-left: 4px solid #ffc107;
            margin: 15px 0;
        }
    </style>
    """
    
    # Vollstandiges HTML
    full_html = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="utf-8">
        {css}
    </head>
    <body>
        {html_content}
    </body>
    </html>
    """
    
    # Bilder einbetten
    full_html = embed_images_in_html(full_html, plots_dir)
    
    # HTML speichern (fur Debug)
    html_path = pdf_path.replace('.pdf', '.html')
    with open(html_path, 'w', encoding='utf-8') as f:
        f.write(full_html)
    print(f"  HTML: {html_path}")
    
    # Create PDF with weasyprint
    try:
        from weasyprint import HTML
        HTML(string=full_html).write_pdf(pdf_path)
        print(f"  PDF: {pdf_path}")
        return True
    except Exception as e:
        print(f"  Fehler bei PDF-Erstellung: {e}")
        return False


def main():
    print("="*60)
    print("PDF-GENERATOR")
    print("="*60)
    
    base_dir = Path(".")
    plots_dir = base_dir / "reports" / "plots"
    
    # Deutsche Version
    print("\n1. Deutsche Dokumentation...")
    md_to_pdf(
        base_dir / "reports" / "Projektdokumentation_DE.md",
        str(base_dir / "reports" / "Projektdokumentation_DE.pdf"),
        str(plots_dir)
    )
    
    # Englische Version
    print("\n2. Englische Dokumentation...")
    md_to_pdf(
        base_dir / "reports" / "Project_Documentation_EN.md",
        str(base_dir / "reports" / "Project_Documentation_EN.pdf"),
        str(plots_dir)
    )
    
    print("\n" + "="*60)
    print("FERTIG!")
    print("="*60)

## ▶ Execution

In [ ]:
main()